In [31]:
import pandas as pd
df=pd.read_csv('different_models_experiment_data.csv')

In [32]:
df.drop('Unnamed: 0',axis=1,inplace=True)

In [33]:
df['loan_status'].value_counts()

loan_status
0    1076751
1     268559
Name: count, dtype: int64

In [34]:
from sklearn.model_selection import train_test_split

X = df.drop("loan_status", axis=1)
y = df["loan_status"]


In [35]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)

In [36]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
X_train['fico_category'] = le.fit_transform(X_train['fico_category'].astype(str))
X_test['fico_category'] = le.transform(X_test['fico_category'].astype(str))

# Verify
print(X_train['fico_category'].unique())  # Should be numbers now
print(X_train.dtypes[X_train.dtypes == 'object'])

[2 3 1 0]
Series([], dtype: object)


In [37]:

class_counts = y_train.value_counts()
print("Class counts in y_train:\n", class_counts)

majority_class = class_counts.idxmax()
minority_class = class_counts.idxmin()

print(f"\nMajority class: {majority_class}")
print(f"Minority class: {minority_class}")

Class counts in y_train:
 loan_status
0    861401
1    214847
Name: count, dtype: int64

Majority class: 0
Minority class: 1


In [38]:

X_train_majority = X_train[y_train == majority_class]
y_train_majority = y_train[y_train == majority_class]

X_train_minority = X_train[y_train == minority_class]
y_train_minority = y_train[y_train == minority_class]

print(f"Shape of X_train_majority: {X_train_majority.shape}")
print(f"Shape of y_train_majority: {y_train_majority.shape}")
print(f"Shape of X_train_minority: {X_train_minority.shape}")
print(f"Shape of y_train_minority: {y_train_minority.shape}")
     

Shape of X_train_majority: (861401, 36)
Shape of y_train_majority: (861401,)
Shape of X_train_minority: (214847, 36)
Shape of y_train_minority: (214847,)


In [39]:

minority_count = len(y_train_minority)

## Random forest dataset

In [40]:

# RandomForest Dataset
X_rf_majority_undersampled = X_train_majority.sample(n=minority_count, random_state=42)
y_rf_majority_undersampled = y_train_majority.sample(n=minority_count, random_state=42)

X_rf = pd.concat([X_rf_majority_undersampled, X_train_minority], axis=0)
y_rf = pd.concat([y_rf_majority_undersampled, y_train_minority], axis=0)

print("RandomForest Dataset Class Counts:")
print(y_rf.value_counts())

RandomForest Dataset Class Counts:
loan_status
0    214847
1    214847
Name: count, dtype: int64


## XGB Dataset

In [41]:

X_xg_majority_undersampled = X_train_majority.sample(n=minority_count, random_state=1)
y_xg_majority_undersampled = y_train_majority.sample(n=minority_count, random_state=1)

X_xg = pd.concat([X_xg_majority_undersampled, X_train_minority], axis=0)
y_xg = pd.concat([y_xg_majority_undersampled, y_train_minority], axis=0)

print("XGBoost Dataset Class Counts:")
print(y_xg.value_counts())

XGBoost Dataset Class Counts:
loan_status
0    214847
1    214847
Name: count, dtype: int64


### LGBM Dataset

In [42]:

X_lgbm_majority_undersampled = X_train_majority.sample(n=minority_count, random_state=2)
y_lgbm_majority_undersampled = y_train_majority.sample(n=minority_count, random_state=2)

X_lgbm = pd.concat([X_lgbm_majority_undersampled, X_train_minority], axis=0)
y_lgbm = pd.concat([y_lgbm_majority_undersampled, y_train_minority], axis=0)

print("LightGBM Dataset Class Counts:")
print(y_lgbm.value_counts())

LightGBM Dataset Class Counts:
loan_status
0    214847
1    214847
Name: count, dtype: int64


In [43]:
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier
rf_base=RandomForestClassifier()
rf_base
     

RandomForestClassifier()

In [44]:

rf_base.fit(X_rf,y_rf)

RandomForestClassifier()

In [45]:

y_pred_rfb_proba = rf_base.predict_proba(X_test)

In [46]:

y_pred_rfb = (y_pred_rfb_proba[:, 1] > 0.5).astype(int)

In [47]:
from sklearn.metrics import classification_report,roc_auc_score
print(classification_report(y_test, y_pred_rfb))
      

              precision    recall  f1-score   support

           0       0.88      0.64      0.74    215350
           1       0.31      0.66      0.42     53712

    accuracy                           0.64    269062
   macro avg       0.60      0.65      0.58    269062
weighted avg       0.77      0.64      0.68    269062



In [48]:

import xgboost as xgb
xgb_base=xgb.XGBClassifier()
xgb_base

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [49]:
xgb_base.fit(X_xg,y_xg)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [50]:

y_pred_xgb_proba=xgb_base.predict_proba(X_test)

In [51]:
y_pred_xgb=(y_pred_xgb_proba[:, 1] > 0.5).astype(int)

In [52]:
print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       0.89      0.64      0.74    215350
           1       0.32      0.68      0.43     53712

    accuracy                           0.65    269062
   macro avg       0.60      0.66      0.59    269062
weighted avg       0.77      0.65      0.68    269062



In [53]:
df=pd.read_csv("accepted_2007_to_2018Q4.csv",usecols=["loan_amnt","term","int_rate","installment","grade","sub_grade","purpose","emp_length","annual_inc","verification_status","home_ownership","fico_range_low","earliest_cr_line","delinq_2yrs","pub_rec","dti","revol_bal","revol_util","bc_util","issue_d","application_type","loan_status","recoveries"])
     

In [55]:
pd.set_option('display.max_columns',None)
df

,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,purpose,dti,delinq_2yrs,earliest_cr_line,fico_range_low,pub_rec,revol_bal,revol_util,recoveries,application_type,bc_util
0,3600.0,36 months,13.99,123.03,C,C4,10+ years,MORTGAGE,55000.0,Not Verified,Dec-2015,Fully Paid,debt_consolidation,5.91,0.0,Aug-2003,675.0,0.0,2765.0,29.7,0.0,Individual,37.2
1,24700.0,36 months,11.99,820.28,C,C1,10+ years,MORTGAGE,65000.0,Not Verified,Dec-2015,Fully Paid,small_business,16.06,1.0,Dec-1999,715.0,0.0,21470.0,19.2,0.0,Individual,27.1
2,20000.0,60 months,10.78,432.66,B,B4,10+ years,MORTGAGE,63000.0,Not Verified,Dec-2015,Fully Paid,home_improvement,10.78,0.0,Aug-2000,695.0,0.0,7869.0,56.2,0.0,Joint App,55.9
3,35000.0,60 months,14.85,829.90,C,C5,10+ years,MORTGAGE,110000.0,Source Verified,Dec-2015,Current,debt_consolidation,17.06,0.0,Sep-2008,785.0,0.0,7802.0,11.6,0.0,Individual,12.1
4,10400.0,60 months,22.45,289.91,F,F1,3 years,MORTGAGE,104433.0,Source Verified,Dec-2015,Fully Paid,major_purchase,25.37,1.0,Jun-1998,695.0,0.0,21929.0,64.5,0.0,Individual,77.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2260696,40000.0,60 months,10.49,859.56,B,B3,9 years,MORTGAGE,227000.0,Verified,Oct-2016,Current,debt_consolidation,12.75,7.0,Feb-1995,705.0,0.0,8633.0,64.9,0.0,Individual,66.9
2260697,24000.0,60 months,14.49,564.56,C,C4,6 years,RENT,110000.0,Not Verified,Oct-2016,Charged Off,debt_consolidation,18.30,0.0,Jul-1999,660.0,1.0,17641.0,68.1,0.0,Individual,77.5
2260698,14000.0,60 months,14.49,329.33,C,C4,10+ years,MORTGAGE,95000.0,Verified,Oct-2016,Current,debt_consolidation,23.36,0.0,Jun-1996,660.0,0.0,7662.0,54.0,0.0,Individual,80.7
2260699,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [56]:
df = df[df['loan_status'].isin(['Fully Paid', 'Charged Off'])]

In [57]:

df['loan_status'].value_counts()

loan_status
Fully Paid     1076751
Charged Off     268559
Name: count, dtype: int64